# Episode 9: Structured Output

So far every reply has been free text in `reply.body`. But often you want **data** — a category, a score, a typed record your code can use directly.

**In this episode you'll build:** an agent that returns a validated Pydantic object instead of a string.

## Why structured output

When downstream code needs to *act* on a reply — route a ticket, store a record, branch on a score — parsing free text with string matching is fragile.

`response_schema` fixes this. You give the agent a schema; the agent returns an object that already matches it. No parsing, no regex, no guessing.

## Setup and the schema

A schema is usually a Pydantic model. `Field(description=...)` tells the model what each field means — treat it like a tool docstring.

Pass the model to `response_schema=` on the `Agent`.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

from typing import Annotated

from pydantic import BaseModel, Field

from autogen.beta import Agent
from autogen.beta.config import OpenAIConfig

config = OpenAIConfig(model="gpt-4.1-mini", temperature=0)


class TicketTriage(BaseModel):
    category: Annotated[str, Field(description="billing, bug, or account_access")]
    urgency: Annotated[str, Field(description="low, medium, or high")]
    summary: Annotated[str, Field(description="Max 100 chars", max_length=100)]


agent = Agent(
    "triage",
    prompt="You triage support messages. Be conservative with urgency.",
    config=config,
    response_schema=TicketTriage,
)


## Ask, and read the typed result

With a schema set, you read the result with `await reply.content()` — which validates the model output and returns a real `TicketTriage` instance. You access fields with normal attribute syntax.

In [1]:
reply = await agent.ask("I was double-charged on my invoice and quarter close is blocked!")
triage = await reply.content()

print("type:    ", type(triage).__name__)
print("category:", triage.category)
print("urgency: ", triage.urgency)
print("summary: ", triage.summary)


type:     TicketTriage
category: billing
urgency:  medium
summary:  Double charge on invoice blocking quarter close


## `reply.body` vs `reply.content()`

Both still exist, and they differ:

- **`reply.body`** — the raw model text (here, the JSON the model produced)
- **`await reply.content()`** — the raw text *validated against the schema*, returned as a typed object

If validation fails, `content()` raises (e.g. `pydantic.ValidationError`). You can pass `content(retries=1)` to let the agent try again on a validation failure.

## Schema types you can pass

`response_schema` accepts more than Pydantic models:

| Type | You get back |
|---|---|
| Pydantic `BaseModel` | An instance of the model |
| `dataclass` | An instance of the dataclass |
| Primitive (`int`, `float`, `bool`) | A bare validated value |
| Union (`int \| str`) | One of the alternatives |
| `dict` / `TypedDict` | A validated dict |

## Additional content

**`@response_schema` for custom parsing.** Decorate a function to clamp, clean, or combine fields — e.g. parse a rating and clamp it to 1–5. The function runs as the validator.

**`ResponseSchema(...)`** lets you give the payload an explicit `name` and `description`, which some providers use as a named contract.

**`PromptedSchema(...)`** injects the JSON schema into the system prompt instead of using the provider's native structured-output mode — useful for models that don't support it natively.

**Structured output composes.** It's just another harness slot, so it works alongside tools, middleware, and assembly — an agent can call tools *and* return a typed object.

## What's next

That completes **Group 2 — the Agent Harness**. You can now shape an agent's behavior (middleware), its context (assembly), and its output (structured output).

Next, Group 3 changes the game: instead of one agent with a rich harness, **many autonomous agents** communicating over a Hub.